# Data Definition Language (DDL)

## 1. What is DDL?
---

DDL is a subset of SQL. Up until now you have only been reading data from different tables in the database. 
This is just the R in CRUD (Create, Read, Update, Delete). 

Data Definition Language enables:
* The creation of tables -- (Create)
* The insertion of data -- (Create)
* Altering tables -- (Update)
* Updating data -- (Update)
* Dropping tables -- (Delete)
* Deleting data -- (Delete)




In [1]:
# Imports
from tabulate import tabulate
import duckdb

In [2]:
# Database file path
# DuckDB is an in-process database, so we connect to a local file.
db_path = './webshop_db.duckdb'

In [3]:
# Establish connection to DuckDB
conn = duckdb.connect(db_path)

# Create a cursor-like object to interact with the database
cur = conn.cursor()

### Cursor in SQL

In SQL, a **cursor** is a database object that allows you to interact with the result set of a query in a controlled, row-by-row manner. It provides a pointer to a specific row within the result set, allowing you to process the data incrementally.

---

#### **Key Points of Cursors**:

- **Purpose**: A cursor enables you to retrieve rows from the database one by one or in chunks, and perform operations like updating, inserting, or deleting specific rows as needed.

- **How it Works**:
    - When you execute a query, the result set is generated by the database. 
    - The cursor helps navigate and interact with this result set.
    - You can use the cursor to move through the data row by row or in batches.

- **Common Operations**:
    - **Fetch Rows**: Cursors allow you to fetch one or more rows from the result set.
    - **Iterate Over Rows**: You can iterate through the result set one row at a time, making it easy to process large sets of data incrementally.
    - **Modify Data**: Cursors can be used to update, delete, or insert rows in a transaction.

---

## Create
---
We start of with a completely empty database

In [4]:
cur.execute("SHOW TABLES;")

records = cur.fetchall()

# Fetch column names
col_names = [desc[0] for desc in cur.description]

# Print the records in table format
print(tabulate(records, headers=col_names, tablefmt="grid"))

+--------+
| name   |
+========+
+--------+


Of course an empty database is useless. We aim to create a database for a webshop, thus we would like to store our products in the databse. However, before we can insert our products we need to create a schema (table).

Creating a table is done using the following SQL:
```SQL
CREATE TABLE table_name (
    col_1 data_type_1 optional_constraint_1,
    col_2 data_type_2 optional_constraint_2,
    col_3 data_type_3 optional_constraint_3,
    ...
    multi_column_constraint
);
```

The first parameters are simple: `table_name` will be the name of your table in the database and `col_i` the name of the columns. The `data_type_i`, `optional_constraint_i`, and `multi_column_constraint` fields require more thought however.

### Data Types

We introduce some of the most essential data types in DuckDB here:

**Numeric**
* Integers: `SMALLINT`, `INTEGER`, and `BIGINT`
* Floating point: `FLOAT` and `DOUBLE`
* Fixed point: `DECIMAL(M,D)` where `M` is the total number of digits and `D` the number of digits after the decimal point

**Strings**
* `CHAR(M)`: fixed-length string
* `VARCHAR(M)`: variable-length string
* `BLOB`: binary data
* `TEXT`: large text values

**Date & Time**
* `DATE`: stores a date in `YYYY-MM-DD`
* `TIMESTAMP`: stores date and time

**Others**
* `BOOLEAN`
* `JSON`

Note that exact data types and implementation details differ across DBMSs.

## Constraints

We introduce some of the most essential constraints here:
* `PRIMARY KEY`: sets a column as the primary key
* `UNIQUE`: ensures all values in the column are unique
* `NOT NULL`: disallows `NULL` values
* `FOREIGN KEY REFERENCES table_name(col)`: enforces referential integrity
* `DEFAULT value`: sets a default value if none is provided
* `CHECK (condition)`: adds custom validation logic

Some constraints can also be added after the column definitions when they span multiple columns:
* `PRIMARY KEY (col_1, col_2)`
* `UNIQUE (col_1, col_2)`
* `FOREIGN KEY (col_1, col_2) REFERENCES table_name(col_1, col_2)`

## Creating our table

Now that we have all our building blocks, let's create a table!

We want the following data to be stored:
* The product ID
* The product name
* The product description
* The product price
* The stock

In [5]:
table_def = '''
DROP TABLE IF EXISTS products;

CREATE TABLE products (
  ID INTEGER PRIMARY KEY,
  Name VARCHAR(255) NOT NULL UNIQUE,
  Description TEXT DEFAULT 'No description',
  Price DECIMAL(10, 2) NOT NULL,
  Stock TINYINT
);
'''

cur.execute(table_def)

# Describe

In [6]:
describe_q = '''
DESCRIBE products;
'''

cur.execute(describe_q)
records = cur.fetchall()

col_names = [desc[0] for desc in cur.description]
print(tabulate(records, headers=col_names, tablefmt="grid"))

+---------------+---------------+--------+-------+------------------+---------+
| column_name   | column_type   | null   | key   | default          | extra   |
+===============+===============+========+=======+==================+=========+
| ID            | INTEGER       | NO     | PRI   |                  |         |
+---------------+---------------+--------+-------+------------------+---------+
| Name          | VARCHAR       | NO     | UNI   |                  |         |
+---------------+---------------+--------+-------+------------------+---------+
| Description   | VARCHAR       | YES    |       | 'No description' |         |
+---------------+---------------+--------+-------+------------------+---------+
| Price         | DECIMAL(10,2) | NO     |       |                  |         |
+---------------+---------------+--------+-------+------------------+---------+
| Stock         | TINYINT       | YES    |       |                  |         |
+---------------+---------------+-------

We created our first table, but for now, it is empty:

In [7]:
cur.execute("SELECT * FROM products;")

records = cur.fetchall()

# Fetch column names
col_names = [desc[0] for desc in cur.description]

# Print the records in table format
print(tabulate(records, headers=col_names, tablefmt="grid"))

+------+--------+---------------+---------+---------+
| ID   | Name   | Description   | Price   | Stock   |
+======+========+===============+=========+=========+
+------+--------+---------------+---------+---------+


## Insert Data

We would like to insert data into our table. This can be done using the following syntax
```SQL
INSERT INTO table_name (col_1, col_2, col_3) VALUES
(value_1, value_2, value_3),
(value_1, value_2, value_3),
(value_1, value_2, value_3),
...;
```
As you can see from the syntax, we can choose what columns to insert data into. When an *incomplete* row is inserted, all the columns not selected are filled in with their default values which are dependent on the data types for the column and the constraints placed on the column 

In [8]:
insert = '''
INSERT INTO products (ID, Name, Price, Stock) VALUES
(1, 'product 1', 12.50, 100),
(2, 'product 2', 25.00, 25),
(3, 'product 3', 99.99, 50);
'''

cur.execute(insert)

In [9]:
cur.execute("SELECT * FROM products;")

records = cur.fetchall()

# Fetch column names
col_names = [desc[0] for desc in cur.description]

# Print the records in table format
print(tabulate(records, headers=col_names, tablefmt="grid"))

+------+-----------+----------------+---------+---------+
|   ID | Name      | Description    |   Price |   Stock |
+======+===========+================+=========+=========+
|    1 | product 1 | No description |   12.5  |     100 |
+------+-----------+----------------+---------+---------+
|    2 | product 2 | No description |   25    |      25 |
+------+-----------+----------------+---------+---------+
|    3 | product 3 | No description |   99.99 |      50 |
+------+-----------+----------------+---------+---------+


As you can see the products were successfully inserted and the ID was automatically incremented!

All the data we inserted satisfied the constraints we put on the columns, but what would happen if we broke a constraint?

In [10]:
oor_insert = '''
INSERT INTO products (ID, Name, Description, Price, Stock) VALUES
(4, 'product 4', 'very fancy product', 10.00, 312);
'''

try:
  cur.execute(oor_insert)
except Exception as exc:
  print(f"ERROR: {exc}")

ERROR: Conversion Error: Type INT32 with value 312 can't be cast because the value is out of range for the destination type INT8

LINE 3: (4, 'product 4', 'very fancy product', 10.00, 312);
                                                      ^


# Update
---

There may be times where we want to alter the schema. We may want to add/remove constraints or add new columns.
In DuckDB, common statements include:

```SQL
ALTER TABLE table_name ADD COLUMN col_name data_type;
ALTER TABLE table_name ALTER COLUMN col_name SET DATA TYPE new_data_type;
ALTER TABLE table_name RENAME COLUMN old_col_name TO new_col_name;
ALTER TABLE table_name DROP COLUMN col_name;
```

Business is growing, so we now need to support much larger stock values.

In [11]:
# Increase Stock capacity and retry the previous insert
cur.execute("ALTER TABLE products ALTER COLUMN Stock SET DATA TYPE INTEGER;")

try:
  cur.execute(oor_insert)
except Exception as exc:
  print(f"ERROR: {exc}")

In [12]:
cur.execute("SELECT * FROM products;")

records = cur.fetchall()

# Fetch column names
col_names = [desc[0] for desc in cur.description]

# Print the records in table format
print(tabulate(records, headers=col_names, tablefmt="grid"))

+------+-----------+--------------------+---------+---------+
|   ID | Name      | Description        |   Price |   Stock |
+======+===========+====================+=========+=========+
|    1 | product 1 | No description     |   12.5  |     100 |
+------+-----------+--------------------+---------+---------+
|    2 | product 2 | No description     |   25    |      25 |
+------+-----------+--------------------+---------+---------+
|    3 | product 3 | No description     |   99.99 |      50 |
+------+-----------+--------------------+---------+---------+
|    4 | product 4 | very fancy product |   10    |     312 |
+------+-----------+--------------------+---------+---------+


## Updating the Data

Now that we altered the table, we would like to update product descriptions by product ID. For this we utilise the following code:
```SQL
UPDATE table_name SET
col_1 = value_1,
col_2 = value_2,
...
WHERE condition
```

We can now update our products as follows

In [13]:
cur.execute("UPDATE products SET Description = 'Best-selling premium product' WHERE ID = 1;")
cur.execute("UPDATE products SET Description = 'Budget-friendly everyday product' WHERE ID = 2;")
cur.execute("UPDATE products SET Description = 'Advanced edition with extra features' WHERE ID = 3;")

In [14]:
cur.execute("SELECT * FROM products;")

records = cur.fetchall()

# Fetch column names
col_names = [desc[0] for desc in cur.description]

# Print the records in table format
print(tabulate(records, headers=col_names, tablefmt="grid"))

+------+-----------+--------------------------------------+---------+---------+
|   ID | Name      | Description                          |   Price |   Stock |
+======+===========+======================================+=========+=========+
|    1 | product 1 | Best-selling premium product         |   12.5  |     100 |
+------+-----------+--------------------------------------+---------+---------+
|    2 | product 2 | Budget-friendly everyday product     |   25    |      25 |
+------+-----------+--------------------------------------+---------+---------+
|    3 | product 3 | Advanced edition with extra features |   99.99 |      50 |
+------+-----------+--------------------------------------+---------+---------+
|    4 | product 4 | very fancy product                   |   10    |     312 |
+------+-----------+--------------------------------------+---------+---------+


# Delete
---

It may happen that we want to remove data from our database or maybe even a whole table.

In terms of our simple database, we might want to remove a product once we are not able to sell it anymore. To remove a row we can make use of the following SQL:
```SQL
DELETE FROM table_name WHERE condition;
```

Let's say that we cannot sell product 2 anymore then we can run the following SQL code

In [15]:
cur.execute("DELETE FROM products WHERE ID = 2")

As you can see below, product 2 was removed from the table:

In [16]:
cur.execute("SELECT * FROM products;")

records = cur.fetchall()

# Fetch column names
col_names = [desc[0] for desc in cur.description]

# Print the records in table format
print(tabulate(records, headers=col_names, tablefmt="grid"))

+------+-----------+--------------------------------------+---------+---------+
|   ID | Name      | Description                          |   Price |   Stock |
+======+===========+======================================+=========+=========+
|    1 | product 1 | Best-selling premium product         |   12.5  |     100 |
+------+-----------+--------------------------------------+---------+---------+
|    3 | product 3 | Advanced edition with extra features |   99.99 |      50 |
+------+-----------+--------------------------------------+---------+---------+
|    4 | product 4 | very fancy product                   |   10    |     312 |
+------+-----------+--------------------------------------+---------+---------+


Now that we are not selling product 2 anymore, our most popular product, the webshop unfortunately went out of business and we want to clean it up.

What we can do first is remove all the data in our table using the `TRUNCATE` keyword as follows:

In [17]:
cur.execute("TRUNCATE TABLE products;")

As can be seen below, all data was removed from the table

In [18]:
cur.execute("SELECT * FROM products;")

records = cur.fetchall()

# Fetch column names
col_names = [desc[0] for desc in cur.description]

# Print the records in table format
print(tabulate(records, headers=col_names, tablefmt="grid"))

+------+--------+---------------+---------+---------+
| ID   | Name   | Description   | Price   | Stock   |
+======+========+===============+=========+=========+
+------+--------+---------------+---------+---------+


Finally we would like to remove the table itself as well. For this we can make use of the `DROP` keyword as follows

In [19]:
cur.execute("DROP TABLE products;")

To see that the `products` table was removed we can run `SHOW TABLES` again

In [20]:
cur.execute("SHOW TABLES;")

records = cur.fetchall()

# Fetch column names
col_names = [desc[0] for desc in cur.description]

# Print the records in table format
print(tabulate(records, headers=col_names, tablefmt="grid"))

+--------+
| name   |
+========+
+--------+
